### 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import utils_z
import cityjson_parser as cjpar
import time

In [3]:
conn = utils_z.get_conn("Test20260413", "postgres", "we6666", "localhost", "5432")

In [49]:
city_name = "luxembourg"
city_prefix  = "LU_LU"

### 1. 处理 lod 1 CityGML 数据并导入数据库

#### 1.1 测试：使用 citygml-tools 将1个 lod1 CityGML 转换为 CityJSON

In [4]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [ ]:
test_xml_path = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')

[10:27:39 INFO] Starting citygml-tools.
[10:27:39 INFO] Executing 'to-cityjson' command.
[10:27:40 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.xml.
[10:27:40 INFO] Writing output to file E:\2_data\building_3d_opensource\hamburg\LoD2-DE_HH_2025-03-14\LoD2_32_557_5932_1_HH.json.
[10:27:41 INFO] Total execution time: 01 s.
[10:27:41 INFO] citygml-tools successfully completed.



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml', returncode=0, stdout="[10:27:39 INFO] Starting citygml-tools.\n[10:27:39 INFO] Executing 'to-cityjson' command.\n[10:27:40 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.xml.\n[10:27:40 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\hamburg\\LoD2-DE_HH_2025-03-14\\LoD2_32_557_5932_1_HH.json.\n[10:27:41 INFO] Total execution time: 01 s.\n[10:27:41 INFO] citygml-tools successfully completed.\n", stderr='')

#### 1.2 测试：解析并检查一个 CityJSON 数据

In [ ]:
import json

test_json_path = test_xml_path.replace('.xml', '.json')
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 查看顶层结构
print(data.keys())
#
# 查看第一栋建筑的属性
first_building_id = list(data["CityObjects"].keys())[0]
first_building = data["CityObjects"][first_building_id]
print(f"\n建筑ID：{first_building_id}")
print(f"类型：{first_building['type']}")
print(f"属性：{json.dumps(first_building.get('attributes', {}), indent=2, ensure_ascii=False)}")
print(f"几何类型：{[g['type'] for g in first_building.get('geometry', [])]}")

dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata'])

建筑ID：DEHHALKAJ0000nVM
类型：Building
属性：{
  "creationDate": "2023-02-23T00:00:00+08:00",
  "DatenquelleLage": "1000",
  "DatenquelleBodenhoehe": "1100",
  "Gemeindeschluessel": "02100141",
  "DatenquelleDachhoehe": "1000",
  "DatenquelleGeschossanzahl": "1000",
  "Geometrietyp2DReferenz": "3000",
  "Grundrissaktualitaet": "2025-01-27",
  "measuredHeight": 2.297,
  "function": "31001_2460",
  "roofType": "1000",
  "storeysAboveGround": 1
}
几何类型：['Solid']


In [11]:
first_geom = first_building["geometry"][0]
print(f"几何类型：{first_geom['type']}")
print(f"lod：{first_geom.get('lod')}")

# 看看boundaries结构
boundaries = first_geom["boundaries"]
print(f"faces数量：{len(boundaries[0])}")
print(f"第一个face：{boundaries[0][0]}")

# 看看transform
print(f"\ntransform：{json.dumps(data['transform'], indent=2)}")
print(f"vertices总数：{len(data['vertices'])}")
print(f"前3个顶点：{data['vertices'][:3]}")

几何类型：Solid
lod：2
faces数量：6
第一个face：[[0, 1, 2, 3]]

transform：{
  "scale": [
    0.001,
    0.001,
    0.001
  ],
  "translate": [
    556975.486,
    5931992.369,
    1.289
  ]
}
vertices总数：7113
前3个顶点：[[649337, 96570, 6982], [649355, 101950, 6982], [640848, 101981, 6982]]


In [9]:
buildings = cjpar.parse_cityjson_lod1(test_json_path)

print(f"解析出建筑数：{len(buildings)}")
b = buildings[0]
print(f"ID：{b['building_id']}")
print(f"高度：{b['height']}")
print(f"楼层：{b['floor_count']}")
print(f"功能：{b['function']}")
print(f"2D footprint：{b['geom_2d']}")

解析出建筑数：1110
ID：DEHHALKA10005SAX
高度：6.88
楼层：2
功能：31001_1010
2D footprint：POLYGON ((549374.76 5936958.141, 549386.416 5936958.461, 549386.675 5936949.114, 549377.3200000001 5936948.865999999, 549375.024 5936948.796, 549374.888 5936953.3889999995, 549374.76 5936958.141))


#### 1.3 批量转换，创建建筑lod1数据库表并导入数据

In [ ]:
# xml批量转换为json
batch_json_output_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\lod1_json"
batch_xml_input_dir = "E:\\2_data\\building_3d_opensource\\hamburg\\LoD1-DE_HH_2023-04-01"

xml_files = [f for f in os.listdir(batch_xml_input_dir) if f.endswith(".xml")]
print(f"共{len(xml_files)}个文件待转换")

errors = []
for i, filename in enumerate(xml_files):
    input_path = os.path.join(batch_xml_input_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{batch_json_output_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd)
        if (i + 1) % 50 == 0:
            print(f"进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

In [3]:
# 创建LOD1建筑表
utils_z.run_sql("""
                CREATE TABLE IF NOT EXISTS hamburg_buildings_lod1
                (
                    building_id
                    VARCHAR
                    PRIMARY
                    KEY,
                    geom_2d
                    GEOMETRY
                (
                    Polygon,
                    25832
                ),
                    height FLOAT,
                    floor_count INTEGER,
                    function VARCHAR,
                    roof_type VARCHAR,
                    year_built INTEGER,
                    geom_3d GEOMETRY
                (
                    PolyhedralSurfaceZ,
                    25832
                )
                    );
                """, conn=conn)

utils_z.run_sql("CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx ON hamburg_buildings_lod1 USING GIST (geom_2d);",
                conn=conn)
print("LOD1建筑表创建完成")

LOD1建筑表创建完成


In [16]:
json_files = [f for f in os.listdir(batch_json_output_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待处理")

total = 0
errors = []

for i, filename in enumerate(json_files):
    filepath = os.path.join(batch_json_output_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod1(filepath)
        print(f"\rProcessing {filename}: {len(buildings)} buildings", end='', flush=True)
        count = cjpar.insert_buildings_lod1(buildings, conn, lod1_table="hamburg_buildings_lod1")
        total += count
        if (i + 1) % 50 == 0:
            print()
            print(f"进度：{i + 1}/{len(json_files)}，已入库建筑：{total}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")

print()  # To move to the next line after the loop
print(f"\n完成！共入库建筑：{total}")
print(f"失败文件数：{len(errors)}")

共784个文件待处理
Processing LoD1_32_554_5924_1_HH.json: 916 buildingss进度：50/784，已入库建筑：15328
Processing LoD1_32_557_5926_1_HH.json: 1033 buildings进度：100/784，已入库建筑：38867
Processing LoD1_32_559_5938_1_HH.json: 539 buildingss进度：150/784，已入库建筑：68189
Processing LoD1_32_561_5937_1_HH.json: 779 buildingss进度：200/784，已入库建筑：88981
Processing LoD1_32_563_5933_1_HH.json: 629 buildingss进度：250/784，已入库建筑：116545
Processing LoD1_32_565_5930_1_HH.json: 475 buildingss进度：300/784，已入库建筑：153498
Processing LoD1_32_567_5921_1_HH.json: 268 buildingss进度：350/784，已入库建筑：179724
Processing LoD1_32_568_5944_1_HH.json: 914 buildingss进度：400/784，已入库建筑：207030
Processing LoD1_32_570_5941_1_HH.json: 551 buildingss进度：450/784，已入库建筑：227768
Processing LoD1_32_572_5930_1_HH.json: 305 buildingss进度：500/784，已入库建筑：254366
Processing LoD1_32_573_5947_1_HH.json: 523 buildingss进度：550/784，已入库建筑：291165
Processing LoD1_32_575_5930_1_HH.json: 1 buildingssss进度：600/784，已入库建筑：314181
Processing LoD1_32_576_5945_1_HH.json: 632 buildingss进度：650/784，已入库建筑：

#### 1.4 （Deprecated）更新block表

In [18]:
result = utils_z.run_sql("""
                         SELECT constraint_name
                         FROM information_schema.table_constraints
                         WHERE table_name = 'hamburg_blocks'
                           AND constraint_type = 'PRIMARY KEY';
                         """, conn=conn, fetch=True)

print(result)

[('blocks_pkey',)]


In [19]:
# 1. 添加新字段
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                    ADD COLUMN IF NOT EXISTS city VARCHAR,
                    ADD COLUMN IF NOT EXISTS country VARCHAR,
                    ADD COLUMN IF NOT EXISTS area_m2 FLOAT;
                """, conn=conn)

# 2. 填入城市、国家、面积
utils_z.run_sql("""
                UPDATE hamburg_blocks
                SET city    = 'Hamburg',
                    country = 'Germany',
                    area_m2 = ST_Area(geom);
                """, conn=conn)

# 3. 删除主键约束
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                DROP
                CONSTRAINT blocks_pkey;
                """, conn=conn)

# 4. 把block_id从INTEGER改为VARCHAR，同时替换为新格式
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                ALTER
                COLUMN block_id TYPE VARCHAR
    USING 'DE_HH_' || LPAD(block_id::TEXT, 6, '0');
                """, conn=conn)

# 5. 重新设置主键
utils_z.run_sql("""
                ALTER TABLE hamburg_blocks
                    ADD PRIMARY KEY (block_id);
                """, conn=conn)

print("改造完成")

# 验证
result = utils_z.run_sql("""
                         SELECT block_id, city, country, area_m2
                         FROM hamburg_blocks LIMIT 5;
                         """, conn=conn, fetch=True)

for row in result:
    print(row)

改造完成
('DE_HH_000003', 'Hamburg', 'Germany', 146583.28380672264)
('DE_HH_000004', 'Hamburg', 'Germany', 68077.03149206015)
('DE_HH_000006', 'Hamburg', 'Germany', 42258.92792725274)
('DE_HH_000007', 'Hamburg', 'Germany', 64097.37637555693)
('DE_HH_000155', 'Hamburg', 'Germany', 18986.38463513086)


#### 1.5 街区与建筑空间叠合，记录建筑所属的街区

In [20]:
# 添加block_id字段
utils_z.run_sql("""
                ALTER TABLE hamburg_buildings_lod1
                    ADD COLUMN IF NOT EXISTS block_id VARCHAR;
                """, conn=conn)

# 建立空间索引
utils_z.run_sql("""
                CREATE INDEX IF NOT EXISTS buildings_lod1_geom_idx
                    ON hamburg_buildings_lod1 USING GIST (geom_2d);
                """, conn=conn)

print("字段和索引准备完成，开始空间叠合...")

字段和索引准备完成，开始空间叠合...


In [22]:
# 用重心落点规则批量判定
start_time = time.time()

utils_z.run_sql("""
                UPDATE hamburg_buildings_lod1 b
                SET block_id = (SELECT bl.block_id
                                FROM hamburg_blocks bl
                                WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
                    LIMIT 1
                    );
                """, conn=conn)

end_time = time.time()
elapsed_time = end_time - start_time
print(f"空间叠合完成，耗时：{elapsed_time:.2f}秒")


空间叠合完成，耗时：23.50秒


In [23]:
# 检查数据结果
result = utils_z.run_sql("""
                         SELECT COUNT(*)        AS total,
                                COUNT(block_id) AS matched,
                                COUNT(*)           FILTER (WHERE block_id IS NULL) AS unmatched
                         FROM hamburg_buildings_lod1;
                         """, conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：384629
成功匹配block：354272
未匹配block：30357


#### 1.6 绘制检查

In [3]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import py5

In [6]:
# 随机选取100个有建筑的block
sql_blocks = """
             SELECT block_id, geom
             FROM hamburg_blocks
             WHERE block_id IN (SELECT DISTINCT block_id
                                FROM hamburg_buildings_lod1
                                WHERE block_id IS NOT NULL)
             ORDER BY RANDOM() LIMIT 50; \
             """
gdf_blocks = gpd.read_postgis(sql_blocks, conn, geom_col="geom", crs="EPSG:25832")
blocks = list(gdf_blocks.itertuples(index=False, name='Block'))
print(f"随机选取了 {len(blocks)} 个block")

# 预加载所有建筑数据
buildings_data = {}
for block in blocks:
    block_id = block.block_id
    sql_buildings = f"""
        SELECT building_id, geom_2d
        FROM hamburg_buildings_lod1
        WHERE block_id = '{block_id}';
    """
    gdf_buildings = gpd.read_postgis(sql_buildings, conn, geom_col="geom_2d", crs="EPSG:25832")
    buildings_data[block_id] = list(gdf_buildings.geometry)

# 全局变量
current_index = 0


def all_polygons(g):
    if isinstance(g, Polygon):
        return [g]
    elif isinstance(g, MultiPolygon):
        return list(g.geoms)
    return []


def draw_polygon(poly, is_building=False):
    """绘制单个polygon，根据是否是建筑设置不同颜色"""
    coords = list(poly.exterior.coords)
    if is_building:
        py5.fill(180, 100, 100)  # 建筑：红棕色
        py5.stroke(120, 60, 60)
    else:
        py5.fill(220, 220, 210)  # block：浅灰色
        py5.stroke(80, 80, 80)

    py5.begin_shape()
    for x, y in coords:
        py5.vertex(x, y)
    py5.end_shape(py5.CLOSE)


def draw_current_block():
    global current_index
    if current_index < 0 or current_index >= len(blocks):
        return

    block = blocks[current_index]
    block_id = block.block_id
    block_geom = block.geom

    buildings_geoms = buildings_data[block_id]

    py5.background(240)

    W, H = py5.width, py5.height
    minx, miny, maxx, maxy = block_geom.bounds
    cw = maxx - minx
    ch = maxy - miny
    margin = 0.15
    scale = min((W * (1 - margin)) / cw, (H * (1 - margin)) / ch)
    cx = (minx + maxx) / 2
    cy = (miny + maxy) / 2

    py5.push_matrix()
    py5.translate(W / 2, H / 2)
    py5.scale(scale, -scale)
    py5.translate(-cx, -cy)

    # 先画block轮廓
    py5.stroke_weight(1 / scale)
    for p in all_polygons(block_geom):
        draw_polygon(p, is_building=False)

    # 再画建筑footprint
    py5.stroke_weight(0.5 / scale)
    for geom in buildings_geoms:
        for p in all_polygons(geom):
            draw_polygon(p, is_building=True)

    py5.pop_matrix()

    # 显示当前block信息
    py5.fill(0)
    py5.text_size(16)
    py5.text(f"Block: {block_id} ({current_index + 1}/{len(blocks)})", 10, 20)


def setup():
    py5.size(900, 900)
    draw_current_block()


def draw():
    pass  # 静态绘制


def key_pressed():
    global current_index
    if py5.key == '1':
        current_index = max(0, current_index - 1)
        draw_current_block()
    elif py5.key == '2':
        current_index = min(len(blocks) - 1, current_index + 1)
        draw_current_block()


py5.run_sketch()


随机选取了 50 个block


### 2 处理lod2的CityGML数据

#### 2.1 测试：任选一个lod2的xml并转换为json，然后解析数据做检查

In [37]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

utils_z.run_cmd(f'"{citygml_tools_path}" --version')

citygml-tools version 2.4.1
(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>




CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" --version', returncode=0, stdout='citygml-tools version 2.4.1\n(C) 2018-2026 Claus Nagel <claus.nagel@gmail.com>\n\n', stderr='')

In [39]:
test_xml_path = f"E:\\2_data\\building_3d_opensource\\{city_name}\\lod2_gml\\ACT2023v2_bati3d_Luxembourg.gml"

utils_z.run_cmd(f'"{citygml_tools_path}" to-cityjson {test_xml_path}')

[16:11:30 INFO] Starting citygml-tools.
[16:11:30 INFO] Executing 'to-cityjson' command.
[16:11:31 INFO] [1|1] Processing file E:\2_data\building_3d_opensource\luxembourg\lod2_gml\ACT2023v2_bati3d_Luxembourg.gml.
[16:11:42 WARN] The input file uses unsupported non-CityGML namespaces: http://www.w3.org/2001/SMIL20/Language, http://www.w3.org/2001/SMIL20/, http://www.ascc.net/xml/schematron.
[16:11:42 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.
[16:11:42 INFO] Writing output to file E:\2_data\building_3d_opensource\luxembourg\lod2_gml\ACT2023v2_bati3d_Luxembourg.json.
[16:12:08 INFO] Total execution time: 37 s.
[16:12:08 INFO] citygml-tools finished with 1 warning(s).



CompletedProcess(args='"E:\\0_mylib\\citygml-tools-2.4.1\\citygml-tools.bat" to-cityjson E:\\2_data\\building_3d_opensource\\luxembourg\\lod2_gml\\ACT2023v2_bati3d_Luxembourg.gml', returncode=0, stdout="[16:11:30 INFO] Starting citygml-tools.\n[16:11:30 INFO] Executing 'to-cityjson' command.\n[16:11:31 INFO] [1|1] Processing file E:\\2_data\\building_3d_opensource\\luxembourg\\lod2_gml\\ACT2023v2_bati3d_Luxembourg.gml.\n[16:11:42 WARN] The input file uses unsupported non-CityGML namespaces: http://www.w3.org/2001/SMIL20/Language, http://www.w3.org/2001/SMIL20/, http://www.ascc.net/xml/schematron.\n[16:11:42 INFO] Non-CityGML content is skipped unless a matching ADE extension has been loaded.\n[16:11:42 INFO] Writing output to file E:\\2_data\\building_3d_opensource\\luxembourg\\lod2_gml\\ACT2023v2_bati3d_Luxembourg.json.\n[16:12:08 INFO] Total execution time: 37 s.\n[16:12:08 INFO] citygml-tools finished with 1 warning(s).\n", stderr='')

In [52]:
import json

# test_json_path = test_xml_path.replace('.xml', '.json')
# test_json_path = test_xml_path.replace('.gml', '.json')
test_json_path = r"E:\2_data\building_3d_opensource\luxembourg\lod2_json\ACT2023v2_bati3d_Luxembourg.json"

In [53]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 第一段：顶层结构和对象类型统计
print("顶层keys:", data.keys())

object_counts = {}
for obj_id, obj in data["CityObjects"].items():
    obj_type = obj["type"]
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1
print("\nCityObject类型统计:")
for k, v in sorted(object_counts.items()):
    print(f"  {k}: {v}")

顶层keys: dict_keys(['type', 'version', 'CityObjects', 'transform', 'vertices', 'metadata', 'appearance'])

CityObject类型统计:
  Building: 23373
  BuildingInstallation: 23044


In [42]:
# 第二段：坐标系和transform
print("\nmetadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])


metadata: {'geographicalExtent': [69887.862543, 73166.54713, 230.38, 79504.212916, 82284.075869, 434.448069], 'referenceSystem': 'http://www.opengis.net/def/crs/EPSG/0/2169'}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [69887.863, 73166.547, 230.38]}
第一个顶点原始值: [5396219, 3337057, 75062]


In [54]:
# 第三段：第一个Building/BuildingPart的属性和几何

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        print(f"\n对象ID: {obj_id}")
        print(f"类型: {obj['type']}")
        print(f"属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        for geom in obj.get("geometry", []):
            print(f"几何LOD: {geom.get('lod')}, type: {geom.get('type')}, "
                  f"semantics surfaces数量: {len(geom.get('semantics', {}).get('surfaces', []))}")
        break



对象ID: ACT_f479b95c-bab6-4f70-85c9-7add0a5e3476
类型: Building
属性: {
  "description": "Source: Administration du Cadastre et de la Topographie\n",
  "name": "Commune Luxembourg",
  "creationDate": "2024-04-29T00:00:00+08:00",
  "measuredHeight": 2.86
}
几何LOD: 2, type: Solid, semantics surfaces数量: 6


In [56]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "BuildingInstallation":
        print(f"属性: {obj.get('attributes', {})}")
        for geom in obj.get("geometry", []):
            print(f"LOD: {geom.get('lod')}, type: {geom.get('type')}, "
                  f"semantics: {len(geom.get('semantics', {}).get('surfaces', []))}")
        break

属性: {'relationToConstruction': 'outside'}
LOD: 2, type: MultiSurface, semantics: 6


In [55]:
# 看所有surface类型
surface_types = set()
for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        if str(geom.get("lod")) == "2":
            for s in geom.get("semantics", {}).get("surfaces", []):
                surface_types.add(s.get("type"))
print("surface类型:", surface_types)

# 看LOD分布
lod_counts = {}
for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        lod = str(geom.get("lod"))
        lod_counts[lod] = lod_counts.get(lod, 0) + 1
print("LOD分布:", lod_counts)

surface类型: {'RoofSurface', 'WallSurface', 'GroundSurface'}
LOD分布: {'2': 46515}


In [58]:
test_buildings = cjpar.parse_cityjson_lod2_LU_LU(test_json_path)
print(f"解析出建筑数：{len(test_buildings)}")
b = test_buildings[0] # 第一个
print(f"citygml_id: {b['citygml_id']}")
print(f"floor_count: {b['floor_count']}")
print(f"function: {b['function']}")
print(f"roof_type: {b['roof_type']}")
print(f"year_built: {b['year_built']}")
print(f"高度：{b['height']}")
print(f"2D footprint：{b['geom_2d']}")
print(f"RoofSurface数量：{len(b['surfaces']['RoofSurface'])}")
print(f"WallSurface数量：{len(b['surfaces']['WallSurface'])}")
print(f"GroundSurface数量：{len(b['surfaces']['GroundSurface'])}")

解析出建筑数：23275
citygml_id: ACT_f479b95c-bab6-4f70-85c9-7add0a5e3476
floor_count: 1
function: None
roof_type: None
year_built: None
高度：2.86
2D footprint：POLYGON ((75284.082 76503.604, 75280.286 76502.539, 75282.371 76494.95000000001, 75286.209 76496.027, 75284.082 76503.604))
RoofSurface数量：1
WallSurface数量：4
GroundSurface数量：1


##### 检查数据规范性

In [51]:
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        for geom in obj.get("geometry", []):
            if str(geom.get("lod")) == "2":
                print(f"surfaces定义: {geom.get('semantics', {}).get('surfaces', [])}")
                print(f"values: {geom.get('semantics', {}).get('values', [])[:3]}")
                break
        break

surfaces定义: [{'type': 'RoofSurface', 'id': 'ID_69d20274-7881-4d19-9a99-cc7226cd510b', 'OBJEKTART': 'Offenes Gebaeude', 'HERKUNFT': 'swisstopo', 'HERKUNFT_MONAT': 6, 'HERKUNFT_JAHR': 2019, 'ORIGINAL_HERKUNFT': 'swisstopo', 'ERSTELLUNG_MONAT': 6, 'ERSTELLUNG_JAHR': 2019, 'REVISION_JAHR': 2019, 'REVISION_MONAT': 6, 'DATUM_ERSTELLUNG': '20220517075227', 'DATUM_AENDERUNG': '20220517075227', 'DACH_MAX': 523.01, 'DACH_MIN': 522.55}]
values: [0, 0]


In [58]:
import os
import json

json_dir = r"E:\2_data\building_3d_opensource\zurich\lod2_json"
files = [f for f in os.listdir(json_dir) if f.endswith(".json")][:10]

for fname in files:
    with open(os.path.join(json_dir, fname), "r", encoding="utf-8") as f:
        data = json.load(f)
    
    for obj_id, obj in data["CityObjects"].items():
        if obj["type"] in ("Building", "BuildingPart"):
            for geom in obj.get("geometry", []):
                if str(geom.get("lod")) == "2":
                    n_surfaces = len(geom.get("semantics", {}).get("surfaces", []))
                    values = geom.get("semantics", {}).get("values", [])
                    
                    if not values:
                        print(f"{fname}: surfaces定义数={n_surfaces}, values=空")
                    elif isinstance(values[0], int):
                        print(f"{fname}: surfaces定义数={n_surfaces}, unique values={set(values)}")
                    else:
                        unique_values = set(v for sublist in values for v in (sublist if isinstance(sublist, list) else [sublist]) if v is not None)
                        print(f"{fname}: surfaces定义数={n_surfaces}, unique values={unique_values}")
                    break
        break

swissBUILDINGS3D_3-0_1071-43.json: surfaces定义数=3, unique values={0, 1, 2}
swissBUILDINGS3D_3-0_1091-12.json: surfaces定义数=0, values=空
swissBUILDINGS3D_3-0_1091-14.json: surfaces定义数=0, values=空
swissBUILDINGS3D_3-0_1091-21.json: surfaces定义数=0, values=空
swissBUILDINGS3D_3-0_1091-22.json: surfaces定义数=1, unique values={0}
swissBUILDINGS3D_3-0_1091-23.json: surfaces定义数=0, values=空
swissBUILDINGS3D_3-0_1091-24.json: surfaces定义数=1, unique values={0}
swissBUILDINGS3D_3-0_1091-32.json: surfaces定义数=0, values=空
swissBUILDINGS3D_3-0_1091-33.json: surfaces定义数=1, unique values={0}
swissBUILDINGS3D_3-0_1091-34.json: surfaces定义数=1, unique values={0}


In [59]:
solid_count = 0
multisurface_count = 0
mixed_count = 0

for fname in os.listdir(json_dir):
    if not fname.endswith(".json"):
        continue
    with open(os.path.join(json_dir, fname), "r", encoding="utf-8") as f:
        data = json.load(f)
    
    types_in_file = set()
    for obj_id, obj in data["CityObjects"].items():
        if obj["type"] in ("Building", "BuildingPart"):
            for geom in obj.get("geometry", []):
                if str(geom.get("lod")) == "2":
                    types_in_file.add(geom.get("type"))
    
    if types_in_file == {"Solid"}:
        solid_count += 1
    elif types_in_file == {"MultiSurface"}:
        multisurface_count += 1
    elif len(types_in_file) > 1:
        mixed_count += 1

print(f"纯Solid文件: {solid_count}")
print(f"纯MultiSurface文件: {multisurface_count}")
print(f"混合文件: {mixed_count}")
print(f"总计: {solid_count + multisurface_count + mixed_count}")

纯Solid文件: 0
纯MultiSurface文件: 0
混合文件: 18
总计: 18


#### 2.2 变量

In [59]:
block_table_name = f"{city_name}_blocks"
lod1_table_name = f"{city_name}_buildings_lod1"
lod2_table_name = f"{city_name}_buildings_lod2"
lod2_surface_table_name  = f"{city_name}_building_surfaces_lod2"

target_srid  = 4326
source_srid  = 2169

#### 2.2 创建LOD2建筑表并批量转换、导入数据

##### 2.2.1 批量转换xml/gml -> json (如需)

In [60]:
# 先批量转换XML到CityJSON
lod2_xml_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_gml"
lod2_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"
os.makedirs(lod2_json_dir, exist_ok=True)

In [29]:
# xml_files = [f for f in os.listdir(lod2_xml_dir) if f.endswith(".xml")]
xml_files = [f for f in os.listdir(lod2_xml_dir) if f.endswith(".gml")]
print(f"共{len(xml_files)}个文件待转换")

errors = [] 
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod2_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod2_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

共217个文件待转换
转换进度：50/217
转换进度：100/217
转换进度：150/217
转换进度：200/217
转换完成，失败0个


##### 2.2.2 建表入库

In [83]:
lod2_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"

In [82]:
# 建表前先创建序列
utils_z.run_sql(f"""
    CREATE SEQUENCE IF NOT EXISTS {lod2_table_name}_seq;
""", conn=conn)

utils_z.run_sql(f"""
    CREATE SEQUENCE IF NOT EXISTS {lod2_surface_table_name}_seq;
""", conn=conn)

# LOD2建筑主表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_table_name} (
        building_id     VARCHAR PRIMARY KEY 
                        DEFAULT '{city_prefix}_B_' || LPAD(NEXTVAL('{lod2_table_name}_seq')::TEXT, 7, '0'),
        block_id        VARCHAR,
        citygml_id      VARCHAR,
        geom_2d         GEOMETRY(Polygon, {target_srid}),
        height          FLOAT,
        floor_count     INTEGER,
        function        VARCHAR,
        roof_type       VARCHAR,
        year_built      INTEGER
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_table_name}_geom_idx
    ON {lod2_table_name} USING GIST (geom_2d);
""", conn=conn)

# LOD2 surface子表
utils_z.run_sql(f"""
    CREATE TABLE IF NOT EXISTS {lod2_surface_table_name} (
        surface_id      VARCHAR PRIMARY KEY
                        DEFAULT '{city_prefix}_S_' || LPAD(NEXTVAL('{lod2_surface_table_name}_seq')::TEXT, 8, '0'),
        citygml_id      VARCHAR,
        building_id     VARCHAR REFERENCES {lod2_table_name}(building_id),
        surface_type    VARCHAR,
        geom_3d         GEOMETRY(PolygonZ, {target_srid})
    );
""", conn=conn)

utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS {lod2_surface_table_name}_building_idx
    ON {lod2_surface_table_name} (building_id);
""", conn=conn)

print(city_prefix+" LOD2表创建完成")

LU_LU LOD2表创建完成


In [84]:
import traceback

In [85]:
json_files = [f for f in os.listdir(lod2_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

total = 0
errors = []
building_counter = 1
surface_counter = 1

for i, filename in enumerate(json_files):
    filepath = os.path.join(lod2_json_dir, filename)
    try:
        buildings = cjpar.parse_cityjson_lod2_LU_LU(filepath, target_lod="2")
        count, building_counter, surface_counter = cjpar.insert_buildings_lod2(
            buildings, conn,
            lod2_table=lod2_table_name,
            surface_table=lod2_surface_table_name,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % 50 == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共1个文件待入库

完成！共入库建筑：23275，共入库表面：764173
失败文件数：0


#### 2.3 更新block_id字段并空间叠合

In [86]:
# 建立空间索引
utils_z.run_sql(f"""
    CREATE INDEX IF NOT EXISTS buildings_lod2_geom_idx
    ON {lod2_table_name} USING GIST (geom_2d);
""", conn=conn)

print("开始空间叠合...")

开始空间叠合...


In [87]:
utils_z.run_sql(f"""
    UPDATE {lod2_table_name} b
    SET block_id = (
        SELECT bl.block_id
        FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
        LIMIT 1
    );
""", conn=conn)
print("空间叠合完成")

空间叠合完成


In [88]:
# 检查匹配情况
result = utils_z.run_sql(f"""
    SELECT
        COUNT(*) AS total,
        COUNT(block_id) AS matched,
        COUNT(*) FILTER (WHERE block_id IS NULL) AS unmatched
    FROM {lod2_table_name};
""", conn=conn, fetch=True)

print(f"总建筑数：{result[0][0]}")
print(f"成功匹配block：{result[0][1]}")
print(f"未匹配block：{result[0][2]}")

总建筑数：23275
成功匹配block：20845
未匹配block：2430


In [89]:
# 检查包含LOD2建筑的街区数量
sql_counts = f"SELECT COUNT(DISTINCT block_id) FROM {lod2_table_name} WHERE block_id IS NOT NULL;"
(lod2_count,) = utils_z.run_sql(sql_counts, conn=conn, fetch=True)[0]

print(f"包含 LOD2 建筑的街区数量: {lod2_count}")

包含 LOD2 建筑的街区数量: 507


##### debug

In [ ]:
building_table_name = f"{city_name}_buildings_lod2"
print(utils_z.run_sql(f"""
    SELECT ST_Extent(geom_2d) FROM {building_table_name};
""", conn=conn, fetch=True))

print(utils_z.run_sql(f"""
    SELECT ST_Extent(geom) FROM {block_table_name};
""", conn=conn, fetch=True))

[('BOX(6.028182405295863 49.59310464709656,6.161260372774985 49.67510556902536)',)]
[('BOX(6.080539822221074 49.57142849955256,6.200692299999999 49.64700039953451)',)]


In [72]:
# 取一个有效block
result = utils_z.run_sql(f"""
    SELECT block_id, ST_AsText(geom)
    FROM {block_table_name}
    WHERE block_id IN (
        SELECT DISTINCT block_id FROM {building_table_name}
        WHERE block_id IS NOT NULL
    )
    LIMIT 1;
""", conn=conn, fetch=True)

test_block_id = result[0][0]
test_block_wkt = result[0][1]
print(f"测试block: {test_block_id}")

# 看这个block内已叠合的建筑数
r1 = utils_z.run_sql(f"""
    SELECT COUNT(*) FROM {building_table_name}
    WHERE block_id = '{test_block_id}';
""", conn=conn, fetch=True)
print(f"已叠合建筑数: {r1[0][0]}")

# 看空间上落在这个block内但block_id为NULL的建筑数
r2 = utils_z.run_sql(f"""
    SELECT COUNT(*) FROM {building_table_name} b
    WHERE b.block_id IS NULL
    AND ST_Within(
        ST_Centroid(b.geom_2d),
        (SELECT geom FROM {block_table_name} WHERE block_id = '{test_block_id}')
    );
""", conn=conn, fetch=True)
print(f"空间上在block内但未叠合的建筑数: {r2[0][0]}")

测试block: LU_LU_000002
已叠合建筑数: 9
空间上在block内但未叠合的建筑数: 0


In [73]:
# 未匹配建筑的坐标范围
print(utils_z.run_sql(f"""
    SELECT ST_Extent(geom_2d) FROM {lod2_table_name}
    WHERE block_id IS NULL;
""", conn=conn, fetch=True))

# 已匹配建筑的坐标范围
print(utils_z.run_sql(f"""
    SELECT ST_Extent(geom_2d) FROM {lod2_table_name}
    WHERE block_id IS NOT NULL;
""", conn=conn, fetch=True))

# 未匹配建筑中，centroid落在任意block内的有多少
print(utils_z.run_sql(f"""
    SELECT COUNT(*) FROM {lod2_table_name} b
    WHERE b.block_id IS NULL
    AND EXISTS (
        SELECT 1 FROM {block_table_name} bl
        WHERE ST_Within(ST_Centroid(b.geom_2d), bl.geom)
    );
""", conn=conn, fetch=True))

[('BOX(6.028182405295863 49.59310464709656,6.160144098294744 49.67510556902536)',)]
[('BOX(6.089018853029132 49.596486220284405,6.161260372774985 49.645542751742056)',)]
[(0,)]


In [74]:
# 取几个建筑的centroid坐标
result = utils_z.run_sql(f"""
    SELECT building_id, 
           ST_X(ST_Centroid(geom_2d)) as lon,
           ST_Y(ST_Centroid(geom_2d)) as lat
    FROM {lod2_table_name}
    WHERE block_id IS NOT NULL
    LIMIT 5;
""", conn=conn, fetch=True)

for row in result:
    print(f"{row[0]}: lon={row[1]:.6f}, lat={row[2]:.6f}")

LU_LU_B_0000619: lon=6.118186, lat=49.625749
LU_LU_B_0001103: lon=6.138633, lat=49.633168
LU_LU_B_0001218: lon=6.118843, lat=49.628746
LU_LU_B_0008422: lon=6.112293, lat=49.628745
LU_LU_B_0017425: lon=6.114406, lat=49.621539


In [76]:
# 查看一栋建筑的所有surface类型和对应的Z值范围
result = utils_z.run_sql(f"""
    SELECT surface_type, 
           ST_ZMin(geom_3d) as z_min,
           ST_ZMax(geom_3d) as z_max,
           ST_AsText(ST_Force2D(geom_3d)) as geom_2d
    FROM {lod2_surface_table_name}
    WHERE building_id = (
        SELECT building_id FROM {lod2_table_name} LIMIT 1
    )
    ORDER BY surface_type;
""", conn=conn, fetch=True)

for row in result:
    print(f"type={row[0]}, z_min={row[1]:.2f}, z_max={row[2]:.2f}")

type=GroundSurface, z_min=302.58, z_max=302.58
type=RoofSurface, z_min=305.44, z_max=305.44
type=WallSurface, z_min=302.58, z_max=305.44
type=WallSurface, z_min=302.58, z_max=305.44
type=WallSurface, z_min=302.58, z_max=305.44
type=WallSurface, z_min=302.58, z_max=305.44


In [79]:
import json
import numpy as np

with open(r"E:\2_data\building_3d_opensource\luxembourg\lod2_json\ACT2023v2_bati3d_Luxembourg.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scale     = np.array(data["transform"]["scale"])
translate = np.array(data["transform"]["translate"])
vertices  = np.array(data["vertices"])
real_vertices = vertices * scale + translate

# 取第一个顶点的还原坐标
print("transform还原后第一个顶点（原始坐标系）:", real_vertices[0])

# 再用PostGIS转换这个点到4326
x, y, z = real_vertices[0]
result = utils_z.run_sql(f"""
    SELECT ST_AsText(
        ST_Transform(
            ST_SetSRID(ST_MakePoint({x}, {y}), 2169),
            4326
        )
    );
""", conn=conn, fetch=True)
print("转换到4326:", result[0][0])

# 同时查数据库里第一栋建筑的geom_2d centroid
result2 = utils_z.run_sql(f"""
    SELECT building_id,
           ST_AsText(ST_Centroid(geom_2d))
    FROM {lod2_table_name}
    LIMIT 1;
""", conn=conn, fetch=True)
print("数据库里第一栋建筑centroid:", result2)

transform还原后第一个顶点（原始坐标系）: [75284.082 76503.604   305.442]
转换到4326: POINT(6.102855408509547 49.62313051934634)
数据库里第一栋建筑centroid: [('LU_LU_B_0000001', 'POINT(6.102843791783141 49.62309157326802)')]


In [80]:
# 已匹配建筑的范围
r1 = utils_z.run_sql(f"""
    SELECT ST_Extent(geom_2d) FROM {lod2_table_name}
    WHERE block_id IS NOT NULL;
""", conn=conn, fetch=True)

# 未匹配建筑的范围  
r2 = utils_z.run_sql(f"""
    SELECT ST_Extent(geom_2d) FROM {lod2_table_name}
    WHERE block_id IS NULL;
""", conn=conn, fetch=True)

# block的范围
r3 = utils_z.run_sql(f"""
    SELECT ST_Extent(geom) FROM {block_table_name};
""", conn=conn, fetch=True)

print("已匹配建筑范围:", r1[0][0])
print("未匹配建筑范围:", r2[0][0])
print("block范围:", r3[0][0])

已匹配建筑范围: BOX(6.089018853029132 49.596486220284405,6.161260372774985 49.645542751742056)
未匹配建筑范围: BOX(6.028182405295863 49.59310464709656,6.160144098294744 49.67510556902536)
block范围: BOX(6.080539822221074 49.57142849955256,6.200692299999999 49.64700039953451)


In [81]:
with open(r"E:\2_data\building_3d_opensource\luxembourg\lod2_json\ACT2023v2_bati3d_Luxembourg.json", "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
vertices      = np.array(data["vertices"])
real_vertices = vertices * scale + translate

# 取第一个建筑的GroundSurface顶点
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] == "Building":
        for geom in obj.get("geometry", []):
            if str(geom.get("lod")) == "2":
                # 取第一个face的顶点
                ring = geom["boundaries"][0][0][0]
                coords = [real_vertices[i] for i in ring]
                print("原始坐标系顶点（前3个）:")
                for c in coords[:3]:
                    print(f"  x={c[0]:.3f}, y={c[1]:.3f}")
                
                # 用PostGIS转换，同时试两种轴顺序
                x, y = coords[0][0], coords[0][1]
                r1 = utils_z.run_sql(f"""
                    SELECT ST_AsText(ST_Transform(
                        ST_SetSRID(ST_MakePoint({x}, {y}), 2169), 4326));
                """, conn=conn, fetch=True)
                r2 = utils_z.run_sql(f"""
                    SELECT ST_AsText(ST_Transform(
                        ST_SetSRID(ST_MakePoint({y}, {x}), 2169), 4326));
                """, conn=conn, fetch=True)
                print(f"\nX,Y顺序转换结果: {r1[0][0]}")
                print(f"Y,X顺序转换结果: {r2[0][0]}")
        break

原始坐标系顶点（前3个）:
  x=75284.082, y=76503.604
  x=75286.209, y=76496.027
  x=75282.371, y=76494.950

X,Y顺序转换结果: POINT(6.102855408509547 49.62313051934634)
Y,X顺序转换结果: POINT(6.119744048487474 49.612173966598085)


In [75]:
# 用PostGIS直接检查转换
result = utils_z.run_sql("""
    SELECT ST_AsText(
        ST_Transform(
            ST_SetSRID(ST_MakePoint(75000, 75000), 2169),
            4326
        )
    );
""", conn=conn, fetch=True)
print("2169转4326测试点:", result[0][0])

2169转4326测试点: POINT(6.09894264813424 49.6096093866702)


In [ ]:
# 1. 检查两张表各自的坐标系
print(utils_z.run_sql(f"SELECT Find_SRID('public', '{lod2_table_name}', 'geom_2d');", conn=conn, fetch=True))
print(utils_z.run_sql(f"SELECT Find_SRID('public', '{block_table_name}', 'geom');", conn=conn, fetch=True))

# 2. 看看两张表的坐标范围是否在同一数量级
print(utils_z.run_sql(f"SELECT ST_Extent(geom_2d) FROM {lod2_table_name};", conn=conn, fetch=True))
print(utils_z.run_sql(f"SELECT ST_Extent(geom) FROM {block_table_name};", conn=conn, fetch=True))

# 3. 随便取一栋建筑的centroid，看它落在哪里
print(utils_z.run_sql(f"SELECT ST_AsText(ST_Centroid(geom_2d)) FROM {lod2_table_name} LIMIT 1;", conn=conn, fetch=True))

[(25833,)]
[(0,)]
[('BOX(-10749.27 331389.01,18183.795 353381.93799999997)',)]
[('BOX(589200.6788676552 5330441.177434857,614929.5032929091 5351551.6025656825)',)]
[('POINT(-10212.01 345059.78)',)]


In [ ]:
# 先确认block表建表时的transform目标
# 再用ST_SetSRID强制修正block表的坐标系（如果数据本身是对的只是没登记）
utils_z.run_sql(f"""
    SELECT ST_AsText(geom) FROM {block_table_name} LIMIT 1;
""", conn=conn, fetch=True)


[('POLYGON((589502.8670227258 5342245.796037353,589441.5448576884 5342292.750634647,589406.7026083237 5342314.294902007,589335.6474588497 5342348.668719315,589285.2988223999 5342363.378227262,589251.7038950417 5342371.735374545,589200.6788676552 5342384.600651726,589215.5367276989 5342441.976828558,589222.653586947 5342461.877248411,589250.7802335394 5342501.984416407,589254.2822857167 5342506.986490045,589284.864469873 5342485.096716668,589309.2792616172 5342467.624236941,589489.3078402273 5342338.758993452,589498.2153494882 5342332.539530742,589508.1986397573 5342325.458679183,589518.2171686984 5342318.500689847,589526.9101452322 5342311.755381777,589543.399448982 5342297.18345026,589537.5657849787 5342289.765458014,589502.8670227258 5342245.796037353))',)]

In [ ]:
# 同时看一眼building原始坐标是否合理
utils_z.run_sql(f"""
    SELECT ST_AsText(geom_2d) FROM {lod2_table_name} LIMIT 1;
""", conn=conn, fetch=True)

[('POLYGON((-10215.689999999999 345057.75,-10214.21 345063.36,-10208.33 345061.81,-10209.81 345056.2,-10215.689999999999 345057.75))',)]

In [23]:
with open("E:\\2_data\\building_3d_opensource\\vienna\\lod2_json\\079090.json", "r") as f:
    data = json.load(f)

print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])
print("metadata:", data.get("metadata", {}))

transform: {'scale': [0.001, 0.001, 0.001], 'translate': [-10284.81, 344996.82, 129.24]}
第一个顶点原始值: [277792, 442497, 22964]
metadata: {'geographicalExtent': [-10284.81, 344996.82, 129.24, -9996.12, 345465.06, 157.12], 'referenceSystem': 'http://www.opengis.net/def/crs/EPSG/0/31256'}
